# Supplementary figure: diel courses of the fluxes

Mean diurnal cycles of N2O, CH4; and NEE per treatment over the
measurement period.

**Input** `../data/raw/fluxes_meteo_mgmt.parquet`
**Output** `../figures/*_diel_by_parcel.png`


# LOAD LIBRARIES

In [ ]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from diive.core.io.files import load_parquet

# READ DATA

In [ ]:
# Load data
filename = "../data/raw/fluxes_meteo_mgmt.parquet"
data = load_parquet(filename)
target_fluxes = ['FN2O_L3.3_CUT_50_QCF0', 'FCH4_L3.3_CUT_50_QCF0', 'NEE_L3.3_CUT_50_QCF0']
data = data[target_fluxes + ['parcel', 'parcel_certainty', 'SW_IN_POT']].copy()

# Entire period
start_date = '2023-11-17'
end_date = '2024-08-21'

data = data.loc[start_date:end_date].copy()

data

In [ ]:
data['day_night'] = np.nan
data.loc[data['SW_IN_POT'] > 50,'day_night'] = 1
data.loc[data['SW_IN_POT'] == 0, 'day_night'] = 0

data['2024-04-01':'2024-04-30']['day_night'].plot(x_compat=True);

data

In [ ]:
mask_day = (data['day_night'] == 1)
mask_night = (data['day_night'] == 0)

# Apply the masks and assign to new columns in the DataFrame
for target in target_fluxes:
    data[target + '_day'] = np.where(mask_day, data[target], np.nan)
    data[target + '_night'] = np.where(mask_night, data[target], np.nan)

# Define masks for parcels
mask_A = (data['parcel'] == 'A') & (data['parcel_certainty'] == 'certain')
mask_B = (data['parcel'] == 'B') & (data['parcel_certainty'] == 'certain')
# Add a mask for all data (footprint) including also uncertain parcel attributions
mask_all = (data['parcel_certainty'] == 'certain') | (data['parcel_certainty'] == 'uncertain')
# Plot the time series for all columns
data.plot(figsize=(10, 6), style='.', x_compat=True, subplots=True)
# Display the plot
plt.tight_layout()
plt.show()

In [ ]:
from plot_config import COLORS

COLORS

In [ ]:
def plot_diel_by_parcel(df, start_date, end_date):
    # Setup for the grouped bar chart spacing
    offsets = [-0.2, 0, 0.2] 
    bar_width = 0.2
    
    for target in target_fluxes:
        
        # 1. Create a figure with 2 subplots. 
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True, 
                                       gridspec_kw={'height_ratios': [3, 1]})
        
        # Define your custom colors right here as a third item in each tuple
        masks_to_plot = [
            (mask_A, 'BAU', COLORS['A']),         
            (mask_B, 'VRA', COLORS['B']),         
            (mask_all, 'Footprint', 'grey')  
        ]
        
        # Unpack 'custom_color' alongside the mask and label
        for i, (mask, label, custom_color) in enumerate(masks_to_plot):
            indat = df[mask].loc[start_date:end_date].copy()
            indat['hour'] = indat.index.hour
            
            grouped = indat.groupby('hour')[target]
            mean = grouped.mean()
            std = grouped.std()
            n = grouped.count()
            se = std / np.sqrt(n)
            
            # 2. Plot the main lines on the top axis (ax1) using custom_color
            ax1.plot(mean.index, mean.values, marker='o', label=label, color=custom_color)
            
            # Notice we no longer need p[0].get_color(). We just pass custom_color directly.
            ax1.fill_between(mean.index, mean.values - se, mean.values + se, 
                             alpha=0.2, color=custom_color)
            
            # 3. Plot the counts as bars on the bottom axis (ax2) using custom_color
            ax2.bar(n.index + offsets[i], n.values, width=bar_width, 
                    color=custom_color, alpha=0.7, label=label)
        
        # 4. Formatting the Top Plot (Fluxes)
        ax1.set_ylabel(target)
        ax1.legend(frameon=False)
        
        # 5. Formatting the Bottom Plot (Counts)
        ax2.set_xlabel('Hour of Day')
        ax2.set_ylabel('Data Count')
        ax2.set_xticks(range(0, 24))
        
        # Adjust layout to prevent clipping
        plt.tight_layout()
        plt.savefig(f'../figures/diel_{target}.png', dpi=300)  # Save the figure
        plt.show()

In [ ]:
plot_diel_by_parcel(df=data, start_date='2023-11-17', end_date='2024-08-21')
